# Implementing a Client

The server works; now the **client** side — the piece that lets your application code talk to the MCP server and use its functionality.

> In real projects you'll usually build **either** a client **or** a server, not both. This project builds both so you can see them work together.

> Implement these in **`cli-project/mcp_client.py`** (fill the `TODO`s). That file is `skip-worktree`, so your edits stay local; the finished version is in `cli-project-complete/`.

## Client architecture

Two components:

- **MCP Client** — a custom class *we* write (`MCPClient`) to make the session easier to use.
- **Client Session** — the actual connection to the server, provided by the MCP Python SDK (`ClientSession`).

The session needs careful **resource management** (connections must be cleaned up). That's why we wrap it in our own class that handles cleanup automatically — in this project via an `AsyncExitStack` and `async with` (`__aenter__` / `__aexit__`).

## Where the client fits

Recall the application flow. The CLI code uses the client at **two points**:

1. **Get the list of available tools** to send to Claude.
2. **Execute a tool** when Claude requests it.

Those two jobs map to the two functions we implement next.

## `list_tools()`

Gets all tools the MCP server exposes:

```python
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools
```

Access the session (the connection), call the SDK's built-in `list_tools()`, return `result.tools`.

## `call_tool()`

Executes a specific tool on the server:

```python
async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)
```

Pass the tool name and input (both supplied by Claude) to the server and return the result.

## Testing the client

`mcp_client.py` has a small test harness at the bottom. Run it directly:

```bash
uv run mcp_client.py
```

It connects to your server and prints the available tools — you should see your tool definitions with descriptions and input schemas.

> Windows note: the harness sets `WindowsProactorEventLoopPolicy` so asyncio + stdio subprocesses work correctly.

## Putting it all together

With the client functions implemented, run the full app:

```bash
uv run main.py
```

Ask: *"What is the contents of the report.pdf document?"* Behind the scenes:

1. The app uses the client to get available tools.
2. Tools + your question go to Claude.
3. Claude decides to use `read_doc_contents`.
4. The app uses the client to execute that tool.
5. The result goes back to Claude, who answers you.

The client is the **bridge** between your application logic and the server's functionality — the same `ListTools` / `CallTool` round-trip from the *MCP clients* lesson, now in code.

Next: **Defining resources** — expose the document list and contents as MCP resources.